# 1.2 Data for the Survival Analysis

**For Class 5 (notebook 3.4).** Among people **with diabetes**, how long does it take to develop **chronic kidney disease (CKD)**? To answer a "how long until…" question we need **dates**: when each person was diagnosed with diabetes, whether and when they were diagnosed with CKD, and how long we could follow them. This notebook pulls those dates.

### By the end of this notebook you'll be able to

1. Explain what data a **time-to-event** (survival) analysis needs
2. Explain **censoring**: why we need each person's last visit
3. Pull diagnosis and visit dates for the study group, and save them

| | |
|---|---|
| ⏱️ **Time** | 20–30 minutes (the queries take a few minutes) |
| 📂 **Uses** | the All of Us dataset (BigQuery) |
| 💾 **Saves** | `condition_arranged.csv`, `visit_arranged.csv` |

**Before you start:**

- Run **0.0 setup**. (1.1 isn't required for this notebook.)

> ⚠️ **Keep All of Us data in the Workbench.** Don't download, screenshot, or share participant-level data, or counts of 1 to 20. Clear outputs before you share or commit this notebook (**Edit → Clear Outputs of All Cells**).

## Key concepts: what a survival analysis needs

For each person we need three things:

| Piece | Here | Why |
|---|---|---|
| **Start (time zero)** | the **first diabetes diagnosis** | everyone's clock starts at the same kind of event |
| **Event** | the **first CKD diagnosis**, if there is one | what we're timing |
| **End of follow-up** | the CKD date, or the **last visit** for people without CKD | how long we watched each person |

**Censoring.** Many people never develop CKD *while we can see them*. We don't know what happens after their last visit: they might develop CKD later. We say they're **censored** at their last visit: we know they were CKD-free **at least** until then. Survival methods use this partial information instead of throwing those people away.

```
Person A  ●━━━━━━━━━━━━━✖        diabetes → CKD after 4 years   (event)
Person B  ●━━━━━━━━━━━━━━━━━━○   diabetes → last visit, no CKD  (censored)
          time zero
```

## The queries

Like 1.1, these queries were **generated by the classic Cohort Builder**, and the only change is the dataset path, `` `{CURRENT_DATASET}.table` ``. Both use the **study group** only (people **with** diabetes, `IN`), with the same age, race, ethnicity, and gender rules as 1.1.

| Query | What it returns | Saved as |
|---|---|---|
| Conditions | every **diabetes** and **CKD** diagnosis, with its date | `condition_arranged.csv` |
| Visits | every visit, with its end date | `visit_arranged.csv` |

Notebook **3.4** turns these into one row per person with a start, an event, and an end date.

In [ ]:
source("../aou_setup.R")  # settings, packages, and helper functions

## Query 1: Diabetes and CKD diagnoses

The query keeps diagnoses whose concept is under **diabetes mellitus** (201820) or under one of our **CKD** concepts (the long list of IDs in the `WHERE` clause), and returns the person, the condition's name, and its start date.

In [ ]:
condition_sql <- r"---(
    SELECT
        c_occurrence.person_id,
        c_standard_concept.concept_name as standard_concept_name,
        c_occurrence.condition_start_datetime 
    FROM
        ( SELECT
            * 
        FROM
            `{CURRENT_DATASET}.condition_occurrence` c_occurrence 
        WHERE
            (
                condition_concept_id IN (SELECT
                    DISTINCT c.concept_id 
                FROM
                    `{CURRENT_DATASET}.cb_criteria` c 
                JOIN
                    (SELECT
                        CAST(cr.id as string) AS id       
                    FROM
                        `{CURRENT_DATASET}.cb_criteria` cr       
                    WHERE
                        concept_id IN (201820, 37017104, 37017813, 37018761, 43531559, 43531562, 43531566, 43531577, 43531578, 43531653, 443597, 443601, 443611, 443612, 443614, 44782429, 44782689, 44782690, 44782691, 44782692, 45763854, 45763855, 45769901, 45769903, 45771075, 45773576, 45773688, 46270353, 46271022, 46273636)       
                        AND full_text LIKE '%_rank1]%'      ) a 
                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                        OR c.path LIKE CONCAT('%.', a.id) 
                        OR c.path LIKE CONCAT(a.id, '.%') 
                        OR c.path = a.id) 
                WHERE
                    is_standard = 1 
                    AND is_selectable = 1)
            )  
            AND (
                c_occurrence.PERSON_ID IN (SELECT
                    distinct person_id  
                FROM
                    `{CURRENT_DATASET}.cb_search_person` cb_search_person  
                WHERE
                    cb_search_person.person_id IN (SELECT
                        person_id 
                    FROM
                        `{CURRENT_DATASET}.cb_search_person` p 
                    WHERE
                        DATE_DIFF(CURRENT_DATE, dob, YEAR) - IF(EXTRACT(MONTH FROM dob)*100 + EXTRACT(DAY FROM dob) > EXTRACT(MONTH FROM CURRENT_DATE)*100 + EXTRACT(DAY FROM CURRENT_DATE), 1, 0) BETWEEN 25 AND 50 
                        AND NOT EXISTS (      SELECT
                            'x'      
                        FROM
                            `{CURRENT_DATASET}.death` d      
                        WHERE
                            d.person_id = p.person_id ) ) 
                    AND cb_search_person.person_id IN (SELECT
                        person_id 
                    FROM
                        `{CURRENT_DATASET}.person` p 
                    WHERE
                        ethnicity_concept_id IN (38003564, 38003563) ) 
                    AND cb_search_person.person_id IN (SELECT
                        person_id 
                    FROM
                        `{CURRENT_DATASET}.person` p 
                    WHERE
                        gender_concept_id IN (45878463, 45880669) ) 
                    AND cb_search_person.person_id IN (SELECT
                        person_id 
                    FROM
                        `{CURRENT_DATASET}.person` p 
                    WHERE
                        race_concept_id IN (8527, 8516, 8515) ) 
                    AND cb_search_person.person_id IN (SELECT
                        criteria.person_id 
                    FROM
                        (SELECT
                            DISTINCT person_id, entry_date, concept_id 
                        FROM
                            `{CURRENT_DATASET}.cb_search_all_events` 
                        WHERE
                            (concept_id IN(SELECT
                                DISTINCT c.concept_id 
                            FROM
                                `{CURRENT_DATASET}.cb_criteria` c 
                            JOIN
                                (SELECT
                                    CAST(cr.id as string) AS id       
                                FROM
                                    `{CURRENT_DATASET}.cb_criteria` cr       
                                WHERE
                                    concept_id IN (201820)       
                                    AND full_text LIKE '%_rank1]%'      ) a 
                                    ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                    OR c.path LIKE CONCAT('%.', a.id) 
                                    OR c.path LIKE CONCAT(a.id, '.%') 
                                    OR c.path = a.id) 
                            WHERE
                                is_standard = 1 
                                AND is_selectable = 1) 
                            AND is_standard = 1 )) criteria ) )
            )) c_occurrence 
    LEFT JOIN
        `{CURRENT_DATASET}.concept` c_standard_concept 
            ON c_occurrence.condition_concept_id = c_standard_concept.concept_id
)---"

conditions <- run_query(str_glue(condition_sql))
glimpse(conditions)

**🔍 Read the output:** one row per diagnosis, so the same person appears many times (every time a diagnosis was recorded). `standard_concept_name` is the diagnosis in words.

We sort by person with `arrange()`, which makes the file easier to browse, and save it. Notebook 3.4 works out who has diabetes and CKD, and when.

In [ ]:
condition_arranged <- conditions %>%
  arrange(person_id)

save_data(condition_arranged, "condition_arranged.csv")

### ✍️ Your turn

Which diagnosis names appear most often? (Hint: `count(column, sort = TRUE)` counts each value and sorts from most to least common.)

<details>
<summary><b>Show answer</b></summary>

```r
condition_arranged %>% count(standard_concept_name, sort = TRUE)
```

Diabetes codes (e.g. *Type 2 diabetes mellitus*) dominate. CKD codes appear less often, since only some people with diabetes develop CKD. Notice codes that mention **both**, like *"…kidney disease due to type 2 diabetes"*: in 3.4 they count as both a diabetes and a CKD diagnosis.

</details>

In [ ]:
# Your code here

## Query 2: Visits

Every visit (outpatient, inpatient, emergency, telehealth) for the same people. Notebook 3.4 uses each person's **last** visit as the end of follow-up for anyone who never develops CKD.

In [ ]:
visit_sql <- r"---(
    SELECT
        visit.PERSON_ID,
        v_standard_concept.concept_name as standard_concept_name,
        visit.visit_start_datetime,
        visit.visit_end_datetime 
    FROM
        `{CURRENT_DATASET}.visit_occurrence` visit 
    LEFT JOIN
        `{CURRENT_DATASET}.concept` v_standard_concept 
            ON visit.visit_concept_id = v_standard_concept.concept_id 
    WHERE
        (
            visit_concept_id IN (5083, 581477, 9201, 9202, 9203)
        )  
        AND (
            visit.PERSON_ID IN (SELECT
                distinct person_id  
            FROM
                `{CURRENT_DATASET}.cb_search_person` cb_search_person  
            WHERE
                cb_search_person.person_id IN (SELECT
                    person_id 
                FROM
                    `{CURRENT_DATASET}.cb_search_person` p 
                WHERE
                    DATE_DIFF(CURRENT_DATE, dob, YEAR) - IF(EXTRACT(MONTH FROM dob)*100 + EXTRACT(DAY FROM dob) > EXTRACT(MONTH FROM CURRENT_DATE)*100 + EXTRACT(DAY FROM CURRENT_DATE), 1, 0) BETWEEN 25 AND 50 
                    AND NOT EXISTS (      SELECT
                        'x'      
                    FROM
                        `{CURRENT_DATASET}.death` d      
                    WHERE
                        d.person_id = p.person_id ) ) 
                AND cb_search_person.person_id IN (SELECT
                    person_id 
                FROM
                    `{CURRENT_DATASET}.person` p 
                WHERE
                    ethnicity_concept_id IN (38003564, 38003563) ) 
                AND cb_search_person.person_id IN (SELECT
                    person_id 
                FROM
                    `{CURRENT_DATASET}.person` p 
                WHERE
                    gender_concept_id IN (45878463, 45880669) ) 
                AND cb_search_person.person_id IN (SELECT
                    person_id 
                FROM
                    `{CURRENT_DATASET}.person` p 
                WHERE
                    race_concept_id IN (8527, 8516, 8515) ) 
                AND cb_search_person.person_id IN (SELECT
                    criteria.person_id 
                FROM
                    (SELECT
                        DISTINCT person_id, entry_date, concept_id 
                    FROM
                        `{CURRENT_DATASET}.cb_search_all_events` 
                    WHERE
                        (concept_id IN(SELECT
                            DISTINCT c.concept_id 
                        FROM
                            `{CURRENT_DATASET}.cb_criteria` c 
                        JOIN
                            (SELECT
                                CAST(cr.id as string) AS id       
                            FROM
                                `{CURRENT_DATASET}.cb_criteria` cr       
                            WHERE
                                concept_id IN (201820)       
                                AND full_text LIKE '%_rank1]%'      ) a 
                                ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                OR c.path LIKE CONCAT('%.', a.id) 
                                OR c.path LIKE CONCAT(a.id, '.%') 
                                OR c.path = a.id) 
                        WHERE
                            is_standard = 1 
                            AND is_selectable = 1) 
                        AND is_standard = 1 )) criteria ) )
        )
)---"

visits <- run_query(str_glue(visit_sql))
glimpse(visits)

We keep just the two columns we need. `select()` picks columns. (This query names its ID column `PERSON_ID` in capitals; we'll rename it in 3.4.)

In [ ]:
visit_arranged <- visits %>%
  arrange(PERSON_ID) %>%
  select(PERSON_ID, visit_end_datetime)

save_data(visit_arranged, "visit_arranged.csv")
list.files(DATA_DIR)

**📦 Functions used**

| Function | Package | What it does |
|---|---|---|
| `arrange()` | dplyr | sorts rows (here by person) |
| `select()` | dplyr | keeps only the columns you name |
| `count()` | dplyr | counts rows for each value of a column |

## ⚠️ Common errors

| Problem | Fix |
|---|---|
| *Not found: Table … cb_search_person* | This dataset doesn't have the Cohort Builder tables; see the Data Explorer route in 1.1 |
| The visit query takes a long time | It returns every visit for thousands of people: let it run |

## ✅ Key takeaways

- A survival analysis needs a **start**, an **event**, and an **end of follow-up** for each person
- People without the event are **censored** at their last observation, not dropped
- We saved the raw diagnoses and visits; **3.4** turns them into a time-to-event table

**Next:** **2.0 clean_and_join**.